In [9]:
%cd /home/dani/projects/neuroscience

/home/dani/projects/neuroscience


In [10]:
import torch
import torch.nn as nn
import torchsde
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

In [11]:
class LinearControlSDE(torch.nn.Module):
    noise_type = 'diagonal'
    sde_type = 'ito'
    
    def __init__(self, A, B, control_input: callable, sigma=0.1):
        super().__init__()
        self.A = A
        self.B = B
        self.sigma = sigma
        self.state_size = A.shape[0]
        self.control_size = B.shape[1]
        self.control_input = control_input
        
    def f(self, t, x):
        """Drift function: Ax + Bu"""
        # Get control input at time t
        u = self.control_input(t, x)
        return torch.matmul(x, self.A.T) + torch.matmul(u, self.B.T)
    
    def g(self, t, x):
        """Diffusion function: σ"""
        batch_size = x.shape[0]
        state_diffusion = self.sigma * torch.ones(batch_size, self.state_size)
        return state_diffusion

In [12]:
m, k, c = 1.0, 1.0, 0.5
A = torch.tensor([[0.0, 1.0],[-k/m, -c/m]])
B = torch.tensor([[0.0], [1.0/m]])

Define the (finite-horizon) controllability Gramian for (T>0):
$$
W_c(T)=\int_0^T e^{A\tau} B B^T e^{A^T\tau} d\tau.
$$

In [13]:
def compute_gramian(A, B, T, n_steps=200):
    times = torch.linspace(0, T, n_steps + 1, dtype=A.dtype, device=A.device)
    
    At = A.unsqueeze(0) * times.view(-1, 1, 1)
    Et = torch.vmap(torch.matrix_exp)(At)
    
    BBT = B @ B.T
    Mt = Et @ BBT @ Et.transpose(-2, -1)
    
    # Apply trapezoidal weights: 0.5 for first and last, 1.0 for middle
    weights = torch.ones(n_steps + 1, dtype=A.dtype, device=A.device)
    weights[0], weights[-1] = 0.5, 0.5
    
    Wc = torch.sum(Mt * weights.view(-1, 1, 1), dim=0)
    Wc *= (T / n_steps)
    return Wc

**Kalman (controllability) criterion:** the pair $(A,B)$ is controllable iff the $n\times nm$ controllability matrix
$$
\mathcal{C} = \big[B|AB|A^2B|\dots|A^{n-1}B\big]
$$
has rank $n$.

In this case, the control is given by
$$
u(s)=B^T e^{A^T(T-s)} W_c(T)^{-1} \big(x_{\text{target}}-e^{AT}x_0\big)
$$
with the minimum cost:
$$
J(u) = \frac{1}{2} \int_0^T \|u(t)\|^2 dt = \frac{1}{2} d^{\top}W_{c}(T)^{-1}d,\qquad d:=x_{\text{target}}-e^{AT}x_{0}
$$

In [14]:
class OptimalRoute(nn.Module):
    def __init__(self, A, B, x_target, T, n_steps=200, regularize=1e-9):
        super().__init__()
        self.x_target = x_target.reshape(1, -1)
        self.A = A
        self.B = B
        self.T = T
        self.m_exp = torch.matrix_exp(A * T)

        Wc = compute_gramian(A, B, T, n_steps=n_steps)
        Wc_reg = Wc + regularize * torch.eye(A.shape[0])
        self.Wc_inv = torch.inverse(Wc_reg)

    def forward(self, t, x):
        s = self.T - t
        term = torch.matrix_exp(self.A.T * s)
        u = ((self.x_target - x @ self.m_exp.T) @ self.Wc_inv.T) @ term.T @ self.B
        return u

In [99]:
m, k, c = 1.0, 1.0, 0.5
A = torch.tensor([[0.0, 1.0],[-k/m, -c/m]])
B = torch.tensor([[0.0], [1.0/m]])
t_span = [0.0, 5.0] # Time Span
ts = torch.linspace(t_span[0], t_span[1], 200)

In [234]:
# Initial condition: [position, velocity]
x0 = [torch.randn(20, 2) for _ in range(3)]
u_values = [-1., 0., 1.]  # Possible control inputs
controls = [(lambda t, x, v=vi: torch.full((x.shape[0], B.shape[1]), v)) for vi in u_values] # No control

sdes = [LinearControlSDE(A, B, control_input=control_input, sigma=0.0) for control_input in controls]

# Simulate multiple trajectories
xs = [torchsde.sdeint(sde, x0i, ts, method="euler", dt_min=1e-2) for sde, x0i in zip(sdes, x0)]
us = [control_input(ts, x0i.view(-1,2)).unsqueeze(0).repeat(200, 1, 1) for control_input, x0i in zip(controls, x0)]
phi = torch.concat([torch.cat(xs, dim=1), torch.cat(us, dim=1)], dim=2).view(-1,3)
x_prime = torch.concat([sde.f(0, xsi.view(-1, 2)).view(200, 20, 2) for sde, xsi in zip(sdes, xs)], dim=1).view(-1,2)

In [235]:
dt = ts[1] - ts[0]
G = dt * torch.sum(phi[:, :, None] * phi[:, None, :], dim=0)
H = dt * torch.sum(phi[:, None, :] * x_prime[:, :, None], dim=0)
assert not torch.isclose(torch.linalg.det(G),torch.tensor([0.0])), "G is singular!"

In [236]:
theta = H @ torch.inverse(G)
assert torch.allclose(theta, torch.cat([A, B], dim=1), atol=2e-1), "The parameters were not recovered accurately."

In [ ]:
ts_np = ts.detach().numpy()
# shuffle order the trajectories for better visualization
xs_np = phi.view(200, 60, -1)[:, torch.randperm(60)].squeeze().detach().numpy()
margin = 0.5
x_min, x_max = xs_np[:, :, 0].min()-margin, xs_np[:, :, 0].max()+margin
y_min, y_max = xs_np[:, :, 1].min()-margin, xs_np[:, :, 1].max()+margin

X1, X2 = np.meshgrid(np.linspace(x_min, x_max, 20), np.linspace(y_min, y_max, 20))
Anp = A.detach().numpy()
U = Anp[0, 0] * X1 + Anp[0, 1] * X2
V = Anp[1, 0] * X1 + Anp[1, 1] * X2

fig, ax = plt.subplots(figsize=(8, 6))
ax.streamplot(X1, X2, U, V, color='lightgray', density=1.5, linewidth=0.5, arrowsize=1)

# Create 5 lines and 5 dots with different colors
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
lines = []
dots = []
for i in range(5):
    line, = ax.plot([], [], '--', color=colors[i], linewidth=1.5)
    dot, = ax.plot([], [], 'o', markersize=8, color=colors[i])
    lines.append(line)
    dots.append(dot)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()

def update(frame):
    artists = []
    for i in range(5):
        lines[i].set_data(xs_np[:frame+1, i, 0], xs_np[:frame+1, i, 1])
        dots[i].set_data([xs_np[frame, i, 0]], [xs_np[frame, i, 1]])
        artists.extend([lines[i], dots[i]])
    return artists

anim = FuncAnimation(fig, update, frames=len(ts_np), blit=True)
anim.save('images/animation_dataset_v2.gif', writer=PillowWriter(fps=20))
plt.close()